## 5. 最小可运行示例：`query()`

> 来源：[Quickstart](https://code.claude.com/docs/en/agent-sdk/quickstart)、[Agent SDK reference - Python](https://code.claude.com/docs/en/agent-sdk/python)

先立签名，再跑示例：

In [ ]:
async def query(
    *,
    prompt: str | AsyncIterable[dict[str, Any]],
    options: ClaudeAgentOptions | None = None,
    transport: Transport | None = None,
) -> AsyncIterator[Message]


功能：提交一个任务，默认每次调用**新开一个 session**、不带任何历史记忆（要续接旧会话需在 options 里传 `resume` / `continue_conversation`，§13「会话」），执行过程以异步消息流的形式返回。

入参：

| 参数 | 类型 | 说明 |
|---|---|---|
| `prompt` | `str \| AsyncIterable[dict]` | 任务输入。传 `str` 是单条消息模式；传 async generator 是流式输入模式。
| `options` | `ClaudeAgentOptions \| None` | 唯一的配置入口。省略或传 `None` 等价于 `ClaudeAgentOptions()` 全默认（当前目录、默认权限模式、最小系统提示）。全部字段按七组功能整理在 §8「配置总线 ClaudeAgentOptions」 |
| `transport` | `Transport \| None` | 自定义与 CLI 子进程的通信通道，常规使用不传 |

出参：`AsyncIterator[Message]`——不是一段字符串，是逐条 yield 的消息流，用 `async for` 消费、按 `isinstance` 分流。`Message` 是 `UserMessage | AssistantMessage | SystemMessage | ResultMessage | StreamEvent | RateLimitEvent` 六种消息类型的 union，每种怎么读是 §7「读懂消息流」 的主题。

下面跑第一个示例。第一次上手只开 `Read`，任务做成"读项目并总结"。先把消息流看懂，再谈写文件、跑 Bash。`prompt` 给的是一个**任务**，不是一句问答；`options` 里只设三个字段：

- `cwd`：Claude 站在哪个目录下做事；只要涉及读/改代码库就显式设。
- `allowed_tools`：**自动批准名单**（不是"全部可用工具列表"）。名单外的工具会走权限流程。
- `max_turns`：最多几轮，防任务无限延长，控成本。

工具组合直接决定 agent 能力档位：

| tools 组合 | agent 能做什么 |
|---|---|
| `Read`, `Glob`, `Grep` | 只读分析 |
| `Read`, `Edit`, `Glob` | 分析并修改代码 |
| `Read`, `Edit`, `Bash`, `Glob`, `Grep` | 完全自动化（改代码 + 跑测试） |

In [2]:
from claude_agent_sdk import query, ClaudeAgentOptions


async def demo_query():
    options = ClaudeAgentOptions(
        cwd=".",
        max_turns=3,
        allowed_tools=["Read", "Glob"],  # 只自动批准读文件 / 找文件
    )
    async for message in query(
        prompt="Read the current project and summarize the main modules.",
        options=options,
    ):
        print(message)


# Jupyter 内核自带事件循环，直接 top-level await；脚本里改用 anyio.run(demo_query)
await demo_query()

HookEventMessage(subtype='hook_started', data={'type': 'system', 'subtype': 'hook_started', 'hook_id': 'f8b634f7-9691-47fe-adf0-a4aed8505701', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'uuid': '149b34ea-c8e4-428b-ad6c-4b107bdc2064', 'session_id': 'd7dc64fe-c1e5-4bf4-ac87-af7db80a15bb'}, hook_event_name='SessionStart', session_id='d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb', uuid='149b34ea-c8e4-428b-ad6c-4b107bdc2064')
HookEventMessage(subtype='hook_response', data={'type': 'system', 'subtype': 'hook_response', 'hook_id': 'f8b634f7-9691-47fe-adf0-a4aed8505701', 'hook_name': 'SessionStart:startup', 'hook_event': 'SessionStart', 'output': '', 'stdout': '', 'stderr': '', 'exit_code': 0, 'outcome': 'success', 'uuid': 'ed9bf792-9309-4ef4-a07d-299418d04e99', 'session_id': 'd7dc64fe-c1e5-4bf4-ac87-af7db80a15bb'}, hook_event_name='SessionStart', session_id='d7dc64fe-c1e5-4bf4-ac87-af7db80a15bb', uuid='ed9bf792-9309-4ef4-a07d-299418d04e99')
SystemMessage(subtype='init', data={

### 5.1 Quickstart 主干示例：让 agent 改文件

上面的最小示例是只读的。官方 quickstart 的主干示例再进一步：给一个带 bug 的文件，放开 `Edit` + `permission_mode="acceptEdits"`，让 agent 自主完成 读 → 分析 → 改 三步。先造出带 bug 的示例文件，再放 agent 去修——看到 `Done: success` 后回看 `utils.py`，应出现防御性代码：

In [ ]:
%%writefile utils.py
def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)      # bug：空列表除零


def get_user_name(user):
    return user["name"].upper()      # bug：user 为 None 时崩溃

In [3]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
)


async def fix_bugs():
    async for message in query(
        prompt="Find and fix the bugs in utils.py. Make the functions handle edge cases safely.",
        options=ClaudeAgentOptions(
            cwd=".",
            allowed_tools=["Read", "Edit", "Glob"],
            permission_mode="acceptEdits",  # 自动批准文件编辑，agent 才能真的改文件
        ),
    ):
        # 官方 quickstart 的消息分流打印：文字、工具调用、收尾分开呈现
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"[Tool: {block.name}]")
        elif isinstance(message, ResultMessage):
            print(f"Done: {message.subtype}")


await fix_bugs()

[Tool: Glob]
[Tool: Bash]
[Tool: Bash]
当前会话的工作目录里没有 `utils.py`：

- 当前目录是笔记库 `study-notes/AI/agents/claude-agent-sdk`，只有 `.ipynb` 和 `.md` 文件
- 会话的文件访问范围也仅限这个目录，无法搜索其他位置

请告诉我 `utils.py` 的完整路径（或在对应项目目录下重新启动会话），我再来找 bug 并修复。
Done: success
